# DocPrompting Pipeline (NLP Team 19 2026 Version)
Team 19 - UIT
**Repo:** https://github.com/buuhq-uit/team19-docprompting



## 1. Clone Repo Github & Cài đặt Môi trường
Tải repo mà nhóm đã chuẩn hóa (sửa lỗi Dependency, Monkey Patching) và cài đặt các thư viện mới nhất.

In [ ]:
%cd /content
!rm -rf team19-docprompting

# Clone repo đã được fork và fix lỗi thư viện
# !git clone https://github.com/buuhq-uit/team19-docprompting.git
!git clone -b pipeline --single-branch https://github.com/buuhq-uit/team19-docprompting.git


/content
Cloning into 'team19-docprompting'...
remote: Enumerating objects: 265, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 265 (delta 21), reused 37 (delta 12), pack-reused 215 (from 1)
Receiving objects: 100% (265/265), 48.86 MiB | 20.53 MiB/s, done.
Resolving deltas: 100% (95/95), done.


In [ ]:
%cd /content/team19-docprompting
!pip install -r requirements.txt

/content/team19-docprompting


## 2. Liên kết Dữ liệu (Symlink)
Liên kết 2 thư mục `data` và `models` từ Google Drive sang thư mục code để chạy siêu nhanh mà không cần tốn dung lượng copy.

In [ ]:
# connect google drive để thấy thư mục docprompting_assets download sẵn data của tác giả
# 2 file: docprompting_data.zip (129MB) và docprompting_generator_models.zip (5.7GB)
# File Data (129MB): 👉 https://drive.google.com/file/d/1CzNlo8-e4XqrgAME5zHEWEKIQMPga0xl/view
# File Model (5.7GB): 👉 https://drive.google.com/file/d/1NmPMxY1EOWkjM7S8VSKa13DKJmEZ3TqV/view
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Xóa thư mục rỗng mặc định của Github
!rm -rf /content/team19-docprompting/data
!rm -rf /content/team19-docprompting/models
# Tạo link với Google Drive
!ln -s /content/drive/MyDrive/docprompting_assets/data /content/team19-docprompting/data
!ln -s /content/drive/MyDrive/docprompting_assets/models /content/team19-docprompting/models
!ls -la /content/team19-docprompting/models/generator/codet5-base

total 871599
-rw------- 1 root root      1531 Dec  2  2022 config.json
-rw------- 1 root root    294364 Dec  2  2022 merges.txt
-rw------- 1 root root 891641279 Dec  2  2022 pytorch_model.bin
-rw------- 1 root root      3083 Dec  2  2022 special_tokens_map.json
-rw------- 1 root root        25 Dec  2  2022 tokenizer_config.json
-rw------- 1 root root    575045 Dec  2  2022 vocab.json


## 3. Dense Retriever

In [ ]:
# Tìm kiếm Top 10 tài liệu mô tả API Python liên quan nhất cho từng câu hỏi
!PYTHONPATH=/content/team19-docprompting python retriever/simcse/run_inference.py \
  --model_name "neulab/docprompting-codet5-python-doc-retriever" \
  --source_file data/conala/conala_nl.txt \
  --target_file data/conala/python_manual_firstpara.tok.txt \
  --source_embed_save_file data/conala/.tmp/src_embedding \
  --target_embed_save_file data/conala/.tmp/tgt_embedding \
  --sim_func cls_distance.cosine \
  --num_layers 12 \
  --save_file data/conala/retrieval_results.json

{
  "model_name": "neulab/docprompting-codet5-python-doc-retriever",
  "batch_size": 48,
  "source_file": "data/conala/conala_nl.txt",
  "target_file": "data/conala/python_manual_firstpara.tok.txt",
  "source_embed_save_file": "data/conala/.tmp/src_embedding",
  "target_embed_save_file": "data/conala/.tmp/tgt_embedding",
  "save_file": "data/conala/retrieval_results.json",
  "top_k": 200,
  "cpu": false,
  "pooler": "cls",
  "log_level": "verbose",
  "nl_cm_folder": "data/conala/nl.cm",
  "sim_func": "cls_distance.cosine",
  "num_layers": 12,
  "origin_mode": false,
  "oracle_eval_file": "data/conala/cmd_dev.oracle_man.full.json",
  "eval_hit": false,
  "normalize_embed": false,
  "source_idx_file": "data/conala/conala_nl.id",
  "target_idx_file": "data/conala/python_manual_firstpara.tok.id"
}
tokenizer_config.json: 1.30kB [00:00, 714kB/s]
vocab.json: 511kB [00:00, 11.4MB/s]
merges.txt: 294kB [00:00, 54.4MB/s]
special_tokens_map.json: 11.3kB [00:00, 23.9MB/s]
config.json: 1.56kB [00:00

## 4. Chạy FiD Code Generator
Không cần bọc (Monkey patch) mã nguồn như phiên bản cũ vì repo Github đã được sửa tận gốc. Batch size được thiết lập ở mức an toàn (2) cho GPU T4.

In [ ]:
!PYTHONPATH=/content/team19-docprompting python generator/fid/test_reader_simple.py \
    --model_path models/generator/conala.fid.codet5.top10/checkpoint/best_dev/best_dev \
    --tokenizer_name models/generator/codet5-base \
    --eval_data data/conala/fid.cmd_test.codet5.t10.json \
    --per_gpu_batch_size 2 \
    --n_context 10 \
    --name conala.fid.codet5.top10 \
    --checkpoint_dir models/generator \
    --result_tag test_same \
    --main_port 81692

0 - Number of nodes: 1
0 - Node ID        : 0
0 - Local rank     : 0
0 - Global rank    : 0
0 - World size     : 1
0 - GPUs per node  : 1
0 - Multi-node     : False
0 - Multi-GPU      : False
0 - Hostname       : 8a15ac78488f
[04/24/2026 10:28:52] {test_reader_simple.py:133} INFO - load the tokenizer from codet5
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Loading weights: 100% 260/260 [00:00<00:00, 981.98it/s, Materializing param=shared.weight] 
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are 

## 5. Xem Kết Quả Sinh Code

In [ ]:
import json
import os

eval_data_path = '/content/team19-docprompting/data/conala/fid.cmd_test.codet5.t10.json'
id_to_question = {}
if os.path.exists(eval_data_path):
    with open(eval_data_path, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)
        id_to_question = {item['id']: item['question'] for item in eval_data}

result_path = "/content/team19-docprompting/models/generator/conala.fid.codet5.top10/test_results_test_same.json"
if os.path.exists(result_path):
    data = []
    with open(result_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))

    print("Hiển thị 5 kết quả Code đầu tiên:\n")
    for item in data[:5]:
        q_id = item.get('question_id')
        print("--- Ý định NL (Intent):", id_to_question.get(q_id, "Không rõ"))
        print("--- Code Đáp án (Gold):", item.get('gold'))
        print("+++ Code AI (Generated):", item.get('clean_code'))
        print("-" * 60)
else:
    print("File kết quả chưa được tạo ra.")

Hiển thị 5 kết quả Code đầu tiên:

--- Ý định NL (Intent): divide the values with same keys of two dictionary `d1` and `d2`
--- Code Đáp án (Gold): {k: (float(d2[k]) / d1[k]) for k in d2}
+++ Code AI (Generated): {k: d1[k] / d2[k] for k, v in list(d1.items())}
------------------------------------------------------------
--- Ý định NL (Intent): divide values associated with each key in dictionary `d1` from values associated with the same key in dictionary `d2`
--- Code Đáp án (Gold): dict((k, float(d2[k]) / d1[k]) for k in d2)
+++ Code AI (Generated): {d1[k] / d2[k] for k, v in list(d1.items())}
------------------------------------------------------------
--- Ý định NL (Intent): download "http://randomsite.com/file.gz" from http and save as "file.gz"
--- Code Đáp án (Gold): testfile = urllib.request.URLopener() testfile.retrieve('http://randomsite.com/file.gz', 'file.gz')
+++ Code AI (Generated): urllib.request.urlretrieve('http://randomsite.com/file.gz', 'http://randomsite.com/file.gz'

## 6. piple line

In [ ]:
# ──── Block 6: Load Pipeline & Chạy nhiều lần ────
%cd /content/team19-docprompting


/content/team19-docprompting


In [ ]:
# 6.1: Chạy riêng Retriever
!PYTHONPATH=/content/team19-docprompting python team19_retriever.py --top_k 10


  Team 19 - Dense Retriever (CodeT5 SimCSE)

📦 Đang load model: neulab/docprompting-codet5-python-doc-retriever
⚠️  Tokenizer HF Hub lỗi, fallback sang: models/generator/codet5-base
Loading weights: 100% 100/100 [00:00<00:00, 2677.38it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
✅ Model loaded trên device: cuda

📄 Đang load target docs từ: data/conala/python_manual_firstpara.tok.txt
   Số lượng tài liệu: 30755
   ♻️  Sử dụng cache embeddings: data/conala/.tmp/tgt_embedding.npy
   Target embeddings shape: (30755, 768)
   🔍 FAISS index sẵn sàng (30755 vectors)

  📋 KẾT QUẢ RETRIEVE (Lần 1)

🔎 Đang retrieve cho 5 câu hỏi (top_k=10)...
Encoding: 100% 1/1 [00:00<00:00,  2.75it/s]

🔹 Query: sort a list of dictionaries by a value of the 

In [ ]:
# 6.2: Chạy riêng Generator với file JSON sẵn có
!PYTHONPATH=/content/team19-docprompting python team19_generator.py \
    --eval_data data/conala/fid.cmd_test.codet5.t10.json \
    --per_gpu_batch_size 2


  Team 19 - FiD Code Generator (CodeT5)

📦 Đang load model: models/generator/conala.fid.codet5.top10/checkpoint/best_dev/best_dev
   📝 Tokenizer: models/generator/codet5-base
Loading weights: 100% 260/260 [00:00<00:00, 831.84it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `

In [ ]:
# 6.3: Chạy Full Pipeline (end-to-end)
!PYTHONPATH=/content/team19-docprompting python team19_generator_pipeline.py \
    --top_k 10 --n_context 10 --per_gpu_batch_size 2

╔══════════════════════════════════════════════════════════╗
║  Team 19 - DocPrompting Full Pipeline                    ║
║  NL Query → Dense Retriever → FiD Generator → Code      ║
╚══════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BƯỚC 1: Load Dense Retriever
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Team 19 - Dense Retriever (CodeT5 SimCSE)

📦 Đang load model: neulab/docprompting-codet5-python-doc-retriever
⚠️  Tokenizer HF Hub lỗi, fallback sang: models/generator/codet5-base
Loading weights: 100% 100/100 [00:00<00:00, 1769.73it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
✅ Model loaded trên device: cuda

📄 Đang load target docs từ: d

7. res api run

In [ ]:
# Cài thêm
!pip install fastapi uvicorn[standard] pyngrok --quiet
# ngrok.set_auth_token("3CeikPvS2u9aaD4iKjOsIkb2aE2_58toW1WpZaeZUPaA1tEBV")  # Lấy ở https://dashboard.ngrok.com
# Nếu ngrok yêu cầu auth token (miễn phí), thêm dòng này trước ngrok.connect
# ngrok.set_auth_token("YOUR_TOKEN")  # Lấy ở https://dashboard.ngrok.com

In [ ]:
# ──── Start API + Swagger UI trên Colab ────
import os, subprocess, time

# Start server ở background
proc = subprocess.Popen(
    ["python", "team19_generator_pipeline_api.py", "--port", "8000"],
    env={**os.environ, "PYTHONPATH": "/content/team19-docprompting"},
    cwd="/content/team19-docprompting",
)

# Đợi model load xong (khoảng 30-60s)
print("⏳ Đang load models, đợi khoảng 60s...")
time.sleep(60)

# Tạo tunnel với ngrok
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(f"\n✅ Swagger UI: {public_url}/docs")
print(f"✅ API Base:   {public_url}")

⏳ Đang load models, đợi khoảng 60s...

✅ Swagger UI: NgrokTunnel: "https://unsaid-snort-uplifting.ngrok-free.dev" -> "http://localhost:8000"/docs
✅ API Base:   NgrokTunnel: "https://unsaid-snort-uplifting.ngrok-free.dev" -> "http://localhost:8000"
